In [1]:
#this script assign a new column in the gpkg file to each LPG re-sell points, based on the average and number of the rating, assessing the quality.
#it is used for the huff model, to define the attractiveness
#it is assumed 1 for places that have average rating >= 2,5 and at least 4 reviews)
#for all the other it is normalized base on the best (of the remaning)  av.rating* n.reviews

In [ ]:
#1. import
import geopandas as gpd
import pandas as pd
import numpy as np

DATA_DIR = "dataset_big"

input_file = f"{DATA_DIR}/lpg_points_nigeria_final_filtred.gpkg"
gdf = gpd.read_file(input_file, layer='lpg_points')
print(f"Initial points: {len(gdf)}")

Initial points: 2833


In [3]:
# 2. Prepare rating and review columns

gdf['rating'] = pd.to_numeric(gdf['rating'], errors='coerce').fillna(0) #avoid error if still there are places with Nan reviews
gdf['user_ratings_total'] = gdf['user_ratings_total'].fillna(0).astype(int)

# 3. Define the "good" places (attractiveness = 1)

good_mask = (gdf['rating'] >= 2.5) & (gdf['user_ratings_total'] >= 4)
print(f"Places meeting 'good' criteria: {good_mask.sum()}")


Places meeting 'good' criteria: 1771


In [4]:
# 4. Compute attractiveness for all places
attractiveness = np.zeros(len(gdf), dtype=float)

# Good places get 1
attractiveness[good_mask] = 1.0

# For the rest, compute product = rating * number of reviews
other_mask = ~good_mask
product = gdf.loc[other_mask, 'rating'] * gdf.loc[other_mask, 'user_ratings_total']

# Find maximum product among the "other" group
max_product = product.max()
if max_product > 0:
    # Normalize: product / max_product
    attractiveness[other_mask] = (product / max_product).values
else:
    # If all products are zero, set attractiveness to 0 for others
    attractiveness[other_mask] = 0.0

gdf['attractiveness'] = np.round(attractiveness, 4)  # round to 4 decimals

In [ ]:
# 5. Quick summary
print("\nAttractiveness summary:")
print(gdf['attractiveness'].describe(percentiles=[.25, .5, .75, .9, .95]))
print(f"\nZero attractiveness (no reviews or zero product), should be zero if correctly filtred : {(gdf['attractiveness'] == 0).sum()}")

# 6. Save the updated GeoPackage
output_file = output_file = f"{DATA_DIR}/lpg_points_nigeria_attractiveness.gpkg"
gdf.to_file(output_file, driver='GPKG', layer='lpg_points')
print(f"\n✅ Saved updated GeoPackage to: {output_file}")


Attractiveness summary:
count    2833.000000
mean        0.804848
std         0.291486
min         0.066700
25%         0.600000
50%         1.000000
75%         1.000000
90%         1.000000
95%         1.000000
max         1.000000
Name: attractiveness, dtype: float64

Zero attractiveness (no reviews or zero product), should be zero if correctly filtred : 0

✅ Saved updated GeoPackage to: lpg_points_nigeria_attractiveness.gpkg
